# KAVALAN — OCR Fine-tune (TrOCR on handwritten/scanned text)

**Run this as a Kaggle Notebook**, not locally — see `ml/README.md` in the repo for setup steps.

1. Add Input dataset: `landlord/handwriting-recognition` (330K+ handwritten word
   images, each labeled with its text in a CSV — `FILENAME`/`IDENTITY` columns)
2. Settings → Accelerator → GPU T4 x2
3. Run All

Goal: fine-tune `microsoft/trocr-base-handwritten` so it transcribes autopsy report
images/scans at least as well as the current Groq vision call in
`src/app/api/analyze/autopsy/extract-image/route.ts`, then export to ONNX so it can run
locally via `onnxruntime-node` instead of an external API call.

In [ ]:
!pip install -q transformers datasets evaluate jiwer optimum[exporters] optimum-onnx onnx onnxruntime pillow accelerate

## Step 1 — Inspect what Kaggle actually mounted

Kaggle dataset internal layouts vary by uploader even for nominally similar datasets.
**Run this cell first** and read the printed tree before touching Step 2 — adjust the
paths/column names in Step 2 to match what you actually see here.

In [ ]:
import os

INPUT_ROOT = "/kaggle/input"
for root, dirs, files in os.walk(INPUT_ROOT):
    depth = root.replace(INPUT_ROOT, "").count(os.sep)
    if depth > 3:
        continue
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root) or root}/")
    if depth == 3:
        for f in files[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... ({len(files)} files total)")

## Step 2 — Build the (image_path, text) pairs list

`landlord/handwriting-recognition` ships as `train_v2/train/` + `written_name_train_v2.csv`
(and equivalent `validation_v2`/`test_v2` splits) with columns `FILENAME`, `IDENTITY`.
A chunk of rows have `IDENTITY == "UNREADABLE"` or are blank — those get dropped below.

If Step 1's printed tree doesn't match this (uploaders occasionally restructure), adjust
`DATASET_DIR` / `LABELS_FILE` / the column names accordingly — the fallback auto-detection
still runs if the expected file isn't found.

In [ ]:
import glob
import os
import pandas as pd

# Search broadly under /kaggle/input rather than assuming a fixed mount path — Kaggle
# has used different conventions (flat /kaggle/input/<name>/ vs nested
# /kaggle/input/datasets/<owner>/<name>/) depending on how the dataset was attached.
preferred = glob.glob("/kaggle/input/**/written_name_train_v2.csv", recursive=True)
candidates = preferred or (
    glob.glob("/kaggle/input/**/*.csv", recursive=True)
    + glob.glob("/kaggle/input/**/*.tsv", recursive=True)
    + glob.glob("/kaggle/input/**/*.json", recursive=True)
)
print("Candidate label files found:", candidates)
LABELS_FILE = candidates[0] if candidates else None

if not LABELS_FILE:
    raise FileNotFoundError(
        "No labels file auto-detected under /kaggle/input. Check Step 1's output and set "
        "LABELS_FILE manually."
    )

# Everything else resolves relative to the labels file's own directory.
DATASET_DIR = os.path.dirname(LABELS_FILE)
print("Using DATASET_DIR:", DATASET_DIR)

if LABELS_FILE.endswith(".json"):
    df = pd.read_json(LABELS_FILE)
else:
    sep = "\t" if LABELS_FILE.endswith(".tsv") else ","
    df = pd.read_csv(LABELS_FILE, sep=sep)

print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
# TODO: rename these to the actual column names printed above if they differ.
IMAGE_COLUMN = "FILENAME"
TEXT_COLUMN = "IDENTITY"

# This dataset marks illegible samples as "UNREADABLE" and has some blank/NaN labels —
# drop those before training. Harmless no-op for other datasets that don't have this.
if TEXT_COLUMN in df.columns:
    before = len(df)
    df = df.dropna(subset=[TEXT_COLUMN])
    df = df[df[TEXT_COLUMN].astype(str).str.upper() != "UNREADABLE"]
    print(f"Dropped {before - len(df)} unreadable/blank-label rows ({len(df)} remain)")

# Optional: cap dataset size for a faster first run through the pipeline. Set to None
# to use the full dataset once you've confirmed everything works end to end.
MAX_SAMPLES = 20000
if MAX_SAMPLES and len(df) > MAX_SAMPLES:
    df = df.sample(n=MAX_SAMPLES, random_state=42).reset_index(drop=True)
    print(f"Subsampled to {len(df)} rows for a faster first run (MAX_SAMPLES={MAX_SAMPLES})")

# Resolve image column to full paths — try a few common locations.
image_dirs = [d for d in glob.glob(f"{DATASET_DIR}/**/", recursive=True)]


def resolve_image_path(name: str) -> str | None:
    for d in image_dirs:
        candidate = os.path.join(d, name)
        if os.path.isfile(candidate):
            return candidate
    return None


df["resolved_path"] = df[IMAGE_COLUMN].apply(resolve_image_path)
unresolved = df["resolved_path"].isna().sum()
print(f"Resolved {len(df) - unresolved}/{len(df)} image paths")
df = df.dropna(subset=["resolved_path"]).reset_index(drop=True)

from sklearn.model_selection import train_test_split

train_df, eval_df = train_test_split(df, test_size=0.1, random_state=42)
print(f"train={len(train_df)} eval={len(eval_df)}")

## Step 3 — Dataset + processor

In [ ]:
import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import TrOCRProcessor

MODEL_NAME = "microsoft/trocr-base-handwritten"
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)


class OCRDataset(Dataset):
    def __init__(self, df, processor, max_target_length=128):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["resolved_path"]).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        labels = self.processor.tokenizer(
            str(row[TEXT_COLUMN]),
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True,
        ).input_ids
        labels = [
            l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels
        ]
        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}


train_dataset = OCRDataset(train_df, processor)
eval_dataset = OCRDataset(eval_df, processor)
print(len(train_dataset), len(eval_dataset))

## Step 4 — Model + training

`EPOCHS` is deliberately small so a first run fits inside a Kaggle session and proves
the pipeline works end to end. Bump it up (e.g. 10–20) for a real fine-tune once this
completes cleanly.

In [ ]:
from transformers import VisionEncoderDecoderModel

model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id

# Generation-control params must live on generation_config, not config, in recent
# transformers versions — setting them on model.config raises a ValueError at
# generate()-time (which Seq2SeqTrainer calls during evaluation).
model.generation_config.max_length = 128
model.generation_config.early_stopping = True
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.length_penalty = 2.0
model.generation_config.num_beams = 4

In [ ]:
import evaluate

cer_metric = evaluate.load("cer")


def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

EPOCHS = 3  # bump up once the pipeline is confirmed working end to end

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/trocr-checkpoints",
    predict_with_generate=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=torch.cuda.is_available(),
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor,
)

trainer.train()

## Step 5 — Sanity-check a few predictions

In [ ]:
import random

sample_idx = random.sample(range(len(eval_df)), min(5, len(eval_df)))
for i in sample_idx:
    row = eval_df.iloc[i]
    image = Image.open(row["resolved_path"]).convert("RGB")
    pixel_values = processor(image, return_tensors="pt").pixel_values.to(model.device)
    generated_ids = model.generate(pixel_values)
    pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print("GT:  ", row[TEXT_COLUMN])
    print("PRED:", pred_text)
    print("---")

## Step 6 — Save + export to ONNX

Uses `optimum` (Hugging Face's export tooling) since TrOCR is an encoder-decoder model
and needs the encoder/decoder graphs exported together with the merged decoder for
generation to work correctly in `onnxruntime`.

In [ ]:
FINETUNED_DIR = "/kaggle/working/trocr-finetuned"
model.save_pretrained(FINETUNED_DIR)
processor.save_pretrained(FINETUNED_DIR)

# onnx/onnxruntime are required for the exporter's dynamic-shape post-processing step —
# without them the command fails partway through (after writing encoder_model.onnx but
# before the decoder) while still printing as if it succeeded.
!pip install -q optimum[exporters] optimum-onnx onnx onnxruntime

ONNX_DIR = "/kaggle/working/trocr-onnx"
!optimum-cli export onnx --model {FINETUNED_DIR} --task image-to-text-with-past {ONNX_DIR}

print("Done. Download the /kaggle/working/trocr-onnx directory from the Output tab.")